<a href="https://www.kaggle.com/code/shreeyashah/irrigation-need-using-stacking-ensemble-final?scriptVersionId=315529702" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
df_train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
df_train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
X = df_train.iloc[:, 1:-1]
y = df_train.iloc[:, -1]

In [5]:
df_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

tnf = ColumnTransformer([
    ('oe', OrdinalEncoder(), [0, 10, 11, 12, 13, 14, 16, 18])]
                         , remainder = 'passthrough')

pipe = make_pipeline(tnf)
X_tnf = pipe.fit_transform(X)

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_tnf= le.fit_transform(y)

In [9]:
X_test = df_test.iloc[:, 1:]
ids = df_test.iloc[:, 0]

In [10]:
from category_encoders import TargetEncoder

cat_cols_idx = [0, 10, 11, 12, 13, 14, 16, 18]
cat_cols = X.columns[cat_cols_idx]
te = TargetEncoder(cols=cat_cols)

X_tnf = te.fit_transform(X, y_tnf)
X_test_tnf = te.transform(X_test)

In [14]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

xgb = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.2,
    reg_alpha=0.5,
    reg_lambda=2.0,
    tree_method='hist',
    device='cuda'
)

lgbm = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.015,
    num_leaves=128,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    device='gpu'
)


cat = CatBoostClassifier(
    iterations=1500,
    depth=8,
    learning_rate=0.02,
    l2_leaf_reg=5,
    auto_class_weights='Balanced',
    task_type='CPU',
    verbose=0
)

In [15]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

stack = StackingClassifier(
    estimators=[
        ('xgb', xgb),
        ('lgbm', lgbm),
        ('cat', cat)
    ],
    final_estimator=LogisticRegression(
        C=0.5,
        max_iter=2000,
        class_weight='balanced'
    ),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=2
)

In [16]:
stack.fit(X_tnf, y_tnf)

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 19
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 19 dense feature groups (12.02 MB) transferred to GPU in 0.063912 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2706
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[

[09:27:12] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.1min finished


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 19 dense feature groups (9.61 MB) transferred to GPU in 0.028574 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 47.7min finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 52.4min finished


StackingClassifier(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.85,
                                              device='cuda',
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              feature_weights=None, gamm...
                                               min_child_samples=30,
                                               n_estimators=1500,
                                               num_leaves=128, reg_alpha=0.1,
                                               reg_lambda=1.0, subsample=0.8)),
                               ('cat',
                                CatBoostClassifier(auto_class_weights='Balanced', depth=8, iterations=1500, l2_leaf_reg=5, learning_rate=0.02, task_type='CPU', verbose=0))],
                   final_estimator=LogisticRegression(C=0.5,
                                                      class_weight='balanced',
                                                      max_iter=2000),
                   n_jobs=-1, stack_method='predict_proba', verbose=2)

In [17]:
probs = stack.predict_proba(X_test_tnf)
y_pred = probs.argmax(axis=1)

In [18]:
y_pred_labels = le.inverse_transform(y_pred)

In [19]:
df = pd.DataFrame({
    'id': ids,
    'Irrigation_Need': y_pred_labels
})

In [20]:
df.to_csv('submission.csv', index=False)